# Collaborative filtering

Now that we have selected a great content-based algorithm (the logistic regression one), I wanted to make recommendations based on behavior of the other users. To do so, I first tried to implement a model using ALS. However, I had huge issues with the Java Heap memory whereas I have a lot or RAM available on my computer. So I had to give up and implement the collaborative filtering method using Tensorflow.

The aim to predict if a given user will like a video is based on finding vector representations for each user (matrix `W`) and each item (matrix `X`) as well as finding a bias (vector `B`) to approximate the actual watch ratios in order to predict new ones.

## Import librairies

In [1]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd
import sys
sys.path.append('../')
import metrics

MANUAL_RANDOM_SEED=42
EMBEDDINGS_PATH='../embeddings'

In [2]:
user_ratings: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_ratings.pkl')

In [3]:
user_ratings.head()

,user_id,video_id,like,watch_ratio
0,0,3649,1,1.273397
2,0,5262,0,0.107613
6,0,6789,0,0.175398
7,0,6812,1,2.212062
8,0,183,0,0.130492


In [4]:
# here is the number of videos and users
user_ratings['video_id'].nunique(), user_ratings['user_id'].nunique()

(9937, 7176)

## Data preprocessing

We will turn our user ratings matrix above to get the video idx as rows and user idx as columns. As video idx and user idx are not continuous, we need to map every idx to a indexed new column that will be continuous from start to end range. This will enables us to retrieve the real indexes later after doing all the computations. We use `pd.factorize` method to do so.

In [5]:
user_ratings['video_index'] = pd.factorize(user_ratings['video_id'])[0]
user_ratings['user_index'] = pd.factorize(user_ratings['user_id'])[0]

In [6]:
user_ratings.head()

,user_id,video_id,like,watch_ratio,video_index,user_index
0,0,3649,1,1.273397,0,0
2,0,5262,0,0.107613,1,0
6,0,6789,0,0.175398,2,0
7,0,6812,1,2.212062,3,0
8,0,183,0,0.130492,4,0


## Split train/validation

I split the train and validation sets as I did previously in the logistic regression model (recall `content-based-models/logistic_regression` notebook)

In [7]:
grouped_user = user_ratings.groupby('user_id')

ratings_train, ratings_validation = [], []
for user_id, data in grouped_user:
    user_train, user_validation = train_test_split(data, random_state=MANUAL_RANDOM_SEED, stratify=data['like'], test_size=0.1)
    ratings_train.append(user_train)
    ratings_validation.append(user_validation)

ratings_train = pd.concat(ratings_train)
ratings_validation = pd.concat(ratings_validation)

In [8]:
# We can see that we lost 1 video on the training set
# It does not matter now beacause if we use the this model over the entire training matrix
# we will not loose the video. For the purpose of only evaluating the model, it will not matter
# a lot.
ratings_train['video_id'].nunique(), ratings_train['user_id'].nunique()

(9906, 7176)

## Building matrices for inputing the model

We need to create 2 matrices:

- `Y` that will hold the ratings where rows are videos and columns the users
- `R` that will indicates if a user with index `i` has seen a video indexed by `j` in the matrix Y

Those matrices will be the inputs of our Tensorflow model to compute the matrices `W` and `X` and vector `b`.

In [9]:
Y = ratings_train.pivot_table(values='watch_ratio', columns='user_index', index='video_index').fillna(-1)

In [10]:
# We get the matrix where videos are the rows and users the columns
# watch retio is used here as the values where we recall that if it is
# above 0.8, the user seems to like the video
Y.head()

user_index,0,1,2,3,4,5,6,7,8,9,...,7166,7167,7168,7169,7170,7171,7172,7173,7174,7175
video_index,,,,,,,,,,,,,,,,,,,,,
0,-1.000000,1.186068,-1.000000,0.670470,-1.0,0.613969,0.684918,-1.000000,0.933560,1.052912,...,-1.000000,-1.000000,0.970001,0.524800,1.370479,-1.000000,0.764608,-1.000000,0.775927,0.653814
1,0.107613,-1.000000,0.430324,0.836242,-1.0,0.506449,0.712190,-1.000000,-1.000000,1.372914,...,-1.000000,-1.000000,-1.000000,0.657941,-1.000000,4.519853,1.810572,-1.000000,0.780602,0.604072
2,-1.000000,0.679129,-1.000000,0.607598,-1.0,0.311751,1.553629,0.877666,-1.000000,-1.000000,...,-1.000000,0.250923,-1.000000,-1.000000,-1.000000,-1.000000,4.573830,-1.000000,0.400543,0.546469
3,2.212062,1.335011,0.585104,0.781227,-1.0,0.459638,2.914150,0.961969,0.660981,0.203113,...,-1.000000,-1.000000,-1.000000,0.229027,0.341070,0.115306,0.921327,0.451995,0.908464,0.490119
4,0.130492,1.698525,-1.000000,-1.000000,-1.0,0.625410,-1.000000,1.838525,1.663934,2.680000,...,1.309508,4.086885,1.215246,-1.000000,5.647869,-1.000000,0.991475,1.607049,1.448852,-1.000000


In [11]:
# Check the integrity of the data
user_ratings[user_ratings['video_id'] == 42].head()

,user_id,video_id,like,watch_ratio,video_index,user_index
312,0,42,1,1.098951,206,0
33398,17,42,1,1.508029,206,17
38029,20,42,0,0.384944,206,20
66647,36,42,0,0.394226,206,36
103603,58,42,0,0.454655,206,58


In [12]:
# We get the excepted value from the dataframe just above
Y.loc[206, 0]

1.098951081407222

In [13]:
Y.shape

(9906, 7176)

In [14]:
Y = Y.to_numpy()
# Defining R, put a 0 if the user has not seen the video yet, 1 otherwise
R = np.where(Y == -1, 0, 1)

In [15]:
# Normalize the data by computing the mean of the watch ratios
Y_mean = (np.sum(Y * R, axis=1) / (np.sum(R, axis=1) + 1e-12)).reshape(-1, 1)
Y_norm = Y - Y_mean

## Building the model

I will use the model provided in the course to implement the collaborative filtering method. I will use Tensorflow and the provided custom cost function to learn the features vectors.

In [16]:
def cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorised for speed. Uses tensorflow operations to be compatible with custom training loop.

    Parameters
    ----------
    X: np.ndarray (num_movies,num_features)
      matrix of item features
    W: np.ndarray (num_users,num_features)
      matrix of user parameters
    b: np.ndarray (1, num_users)
      vector of user parameters
    Y: np.ndarray (num_movies,num_users)
      matrix of user ratings of movies
    R: np.ndarray (num_movies,num_users)
      matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
    lambda_: float
      regularization parameter

    Returns
    -------
    float
        The value of the cost given the parameters.
    """
    # Enter the vectorised implementation of the cost function here

    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y) * R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_ / 2) * (
        tf.reduce_sum(X**2) + tf.reduce_sum(W**2)
    )
    return J

In [17]:
# Define the input dimensions for the model
num_videos, num_users = Y.shape
# This is the number of features we want for the user and video vectors 
# that will be learned by the model
num_features = 140

tf.random.set_seed(MANUAL_RANDOM_SEED)

# Set Initial Parameters (W, X), use tf.Variable to track these variables
W = tf.Variable(tf.random.normal((num_users, num_features), dtype=tf.float64), name="W")
X = tf.Variable(
    tf.random.normal((num_videos, num_features), dtype=tf.float64), name="X"
)
b = tf.Variable(tf.random.normal((1, num_users), dtype=tf.float64), name="b")

# Instantiate an optimizer.
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-1)

## Training the model

In [18]:
iterations = 300
lambda_ = 0.8
for iter in range(iterations):
    # Use TensorFlow’s GradientTape
    # to record the operations used to compute the cost
    with tf.GradientTape() as tape:
        # Compute the cost (forward pass included in cost)
        cost_value = cost_func(X, W, b, Y_norm, R, lambda_)

    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss
    grads = tape.gradient(cost_value, [X, W, b])

    # Run one step of gradient descent by updating
    # the value of the variables to minimize the loss.
    optimizer.apply_gradients(zip(grads, [X, W, b]))

    # Log periodically.
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 649874633.0
Training loss at iteration 20: 16963543.7
Training loss at iteration 40: 10618244.9
Training loss at iteration 60: 8420359.2
Training loss at iteration 80: 7123607.9
Training loss at iteration 100: 6354498.1
Training loss at iteration 120: 5859452.2
Training loss at iteration 140: 5513583.0
Training loss at iteration 160: 5257117.8
Training loss at iteration 180: 5058886.5
Training loss at iteration 200: 4901369.5
Training loss at iteration 220: 4773842.5
Training loss at iteration 240: 4669116.5
Training loss at iteration 260: 4582090.4
Training loss at iteration 280: 4509044.1


## Evaluation

As I did in previous models, I build the recommendation matrix by predicting the top videos from the model then I will pass the matrix through the metrics functions (Precision@K, NDCG@K and diversity) to measure the performances of the model.

In [19]:
# Build the predictions matrix from the users, videos and bias vectors learned by the model
# It is really important to add back the mean used to normalize the inputs in order
# to get back coherence on the outputs
predictions = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy() + Y_mean

In [20]:
recommendations: pd.DataFrame = None

# Group evaluation ratings by user and loop through them 
for user_index, user_data in ratings_validation.groupby('user_index'):
    # To compute metrics, we only want to predict videos where the user
    # has already seen it because we would not have any comparison data otherwise
    user_validation_videos = user_data['video_index'].unique()
    # Filter users we might do not have because of splitting the train/validation
    # sets
    user_validation_videos = list(filter(lambda x: x < predictions.shape[1], user_validation_videos))

    # Get predictions for the user (by getting the column of that user)
    # filtering then only the video the user has seen in the validation set
    # but that has not been seen during training
    user_predictions = predictions[:, user_index][user_validation_videos]

    # Buildind the predictions dataframe by using column_stack to concat 
    # each video with its prediction by the model
    user_predictions = pd.DataFrame(np.column_stack((user_validation_videos, user_predictions)), columns=['video_index', 'prediction'])

    # Sort by prediction to get the top recommended videos
    user_predictions = user_predictions.sort_values('prediction', ascending=False)

    # Getting back the real data from the validation set
    top_videos = user_data.merge(user_predictions, on='video_index').sort_values('prediction', ascending=False)

    if recommendations is None:
        recommendations = top_videos
    else:
        recommendations = pd.concat([recommendations, top_videos])

In [21]:
recommendations.head()

,user_id,video_id,like,watch_ratio,video_index,user_index,prediction
4,0,1258,1,1.725023,1425,0,44.620680
144,0,6021,0,0.000000,1100,0,43.059937
11,0,10500,1,2.217258,1726,0,36.206849
95,0,8819,1,2.241311,689,0,32.693522
89,0,600,1,4.127774,387,0,31.209646


In [22]:
# Merge recommendations with video content features to compute the diversity metric
video_content: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_content.pkl')
video_tags: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_one_hot_tags.pkl')

recommendations = recommendations.reset_index()
recommendations = recommendations.merge(video_tags, on='video_id')
recommendations = recommendations.merge(video_content.iloc[:, 2:], on='video_id')

recommendations.head() 

,index,user_id,video_id,like,watch_ratio,video_index,user_index,prediction,tag_0,tag_1,...,tag_28,tag_29,tag_30,show_cnt,play_cnt,valid_play_cnt,like_cnt,comment_cnt,share_cnt,video_duration
0,4,0,1258,1,1.725023,1425,0,44.620680,0,0,...,0,0,0,1535399,1411748,762417,120299,4585,752,7467
1,144,0,6021,0,0.000000,1100,0,43.059937,0,0,...,1,0,0,3474051,3585107,2555135,39991,921,78,8174
2,11,0,10500,1,2.217258,1726,0,36.206849,0,0,...,1,0,0,15681762,16258851,11246578,221353,22462,4088,6200
3,95,0,8819,1,2.241311,689,0,32.693522,0,0,...,1,0,0,3228709,3346634,1989078,13040,719,2833,6100
4,89,0,600,1,4.127774,387,0,31.209646,0,0,...,0,0,0,358626,363059,275808,1517,2,1,3334


In [23]:
# Select columns in the recommendation dataframe to compute the diversity metric on
video_content_columns = recommendations.columns[8:]
video_content_columns

Index(['tag_0', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7',
       'tag_8', 'tag_9', 'tag_10', 'tag_11', 'tag_12', 'tag_13', 'tag_14',
       'tag_15', 'tag_16', 'tag_17', 'tag_18', 'tag_19', 'tag_20', 'tag_21',
       'tag_22', 'tag_23', 'tag_24', 'tag_25', 'tag_26', 'tag_27', 'tag_28',
       'tag_29', 'tag_30', 'show_cnt', 'play_cnt', 'valid_play_cnt',
       'like_cnt', 'comment_cnt', 'share_cnt', 'video_duration'],
      dtype='object')

### Measuring metrics

In [24]:
# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)
    diversity = metrics.inter_list_diversity_at_k(recommendations, video_content_columns, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print(f'ILD@{k} -> {diversity}')
    print()

Precision@5 -> 0.6983835005574136
NDCG@5 -> 0.8615834103214898
ILD@5 -> 0.8926782182385974

Precision@10 -> 0.6772853957636568
NDCG@10 -> 0.8624688644797828
ILD@10 -> 0.9017392984459767

Precision@15 -> 0.659689706428837
NDCG@15 -> 0.8615256071432796
ILD@15 -> 0.9085582156833047

Precision@20 -> 0.6423007246376811
NDCG@20 -> 0.8610337558801838
ILD@20 -> 0.9141994507117441



## Interpretation

As we can see, we top up almost 70% in precision using collaborative filtering. This precision is not as good as I thought to be. I think if I would have been able to use ALS I may get better results but I did not have enough time to debug my issue with Java. 

To put into context, the precision and NDCG may get up by adding more and more interactions between users and videos. That is 70% is enough in my opinion because when I will train the model over the entire big matrix (without using a validation set but the whole train set), I think I will get better precision and NDCG because the model will have more interactions to work with.

Moreover, NDCG may also get better when I will combine the predictions of different models.